# Ground curvature via global IRLS quadric

Pipeline for analysing a ground-only LiDAR scan (e.g. a hill):

1. Load a nuScenes-format `.bin` containing already-filtered ground points.
2. Visualize the cloud (top-down + side view + 3D).
3. Fit a single robust quadric surface  $z = a x^2 + b xy + c y^2 + d x + e y + f$  to **all** points using **IRLS (Iteratively Reweighted Least Squares)** with the Tukey biweight loss. This downweights outliers (residual vegetation, noise) automatically, no thresholds needed.
4. Compute mean and Gaussian curvature analytically from the quadric coefficients.
5. Forward 1-D height profile (raw bin medians + the quadric's prediction) — same layout as your friend's `plot_road_profile`.
6. Forward 1-D signed curvature graph along the slice — bottom panel of the same plot, fill-coloured by sign.

Why a *global* quadric instead of per-bin local fits:
- One coherent surface model — the curvature varies smoothly because the geometry is smooth.
- Per-bin curvature graphs (your friend's PCA approach) are noisier because each bin is fit independently; a global quadric trades that responsiveness for noise robustness.
- Only valid if the ground really is roughly quadric in the area you're looking at (one hill, one valley, one saddle, or a planar slope). For terrain with multiple bumps, switch to local IRLS fits — see the note at the bottom.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers 3d projection)

np.set_printoptions(suppress=True, precision=4)

# ============================================================
# CONFIG -- change paths/ranges here, the rest of the notebook
# is identical regardless of the input.
# ============================================================

# Path to the ground-only .bin file (nuScenes layout: float32 (N,5) -> [x,y,z,intensity,ring])
BIN_PATH = r"C:\Users\karel\OneDrive\BEP\Ground points\Station Brug\Station.4.PW.bin"

# ROI for fitting the quadric. None on either bound = use data extent.
# By default Y is cropped to a 2 m wide strip (1 m to each side of
# the bike) so the quadric only sees the slice of ground we actually
# ride over. The X bounds are left open so the full forward extent
# of the scan is used; tighten if your scan reaches further than the
# region you want to model.
ROI_X = (None, None)   # (x_min, x_max) in metres, e.g. (-15, 25)
ROI_Y = (-1.0, 1.0)    # 2 m wide strip centred on the bike (default)

# Forward 1D profile config (matches the friend's plot_road_profile layout):
#   PROFILE_X_RANGE  -- x-extent of the profile [m]
#   PROFILE_Y_HALF   -- thin strip half-width around y = PROFILE_Y_CENTER [m]
#   PROFILE_BIN      -- x-bin width for showing raw bin medians [m]
#   PROFILE_Y_CENTER -- centre of the strip in y; 0.0 = "down the middle"
PROFILE_X_RANGE  = (-5.0, 20.0)
PROFILE_Y_HALF   = 1.0
PROFILE_BIN      = 0.5
PROFILE_Y_CENTER = 0.0

# IRLS tuning. Defaults are sensible for outdoor LiDAR.
IRLS_MAX_ITER  = 30
IRLS_TOL       = 1e-7
IRLS_C_TUKEY   = 4.685    # Tukey biweight constant (95% efficiency at Gaussian)


## 1. Load the `.bin` file

Same loader as your friend's notebook -- nuScenes layout, drop intensity & ring, keep XYZ.

In [ ]:
def load_bin_xyz(path, cols=None):
    """Load (N, 3) XYZ from a .bin file of float32 points.

    cols: 4 (x,y,z,intensity) or 5 (x,y,z,intensity,ring). If None, inferred
          from file size; raises when the size fits both layouts (ambiguous).
    """
    raw = np.fromfile(path, dtype=np.float32)
    if cols is None:
        div4, div5 = raw.size % 4 == 0, raw.size % 5 == 0
        if div4 and div5:
            raise ValueError(
                f"{path}: {raw.size} floats is divisible by both 4 and 5 — "
                f"can't tell the layout. Pass cols=4 or cols=5 explicitly."
            )
        elif div5:
            cols = 5
        elif div4:
            cols = 4
        else:
            raise ValueError(f"{path}: {raw.size} floats not divisible by 4 or 5.")
    elif raw.size % cols != 0:
        raise ValueError(f"{path}: {raw.size} floats not divisible by cols={cols}.")
    return raw.reshape(-1, cols)[:, :3].astype(np.float64)

points_all = load_bin_xyz(BIN_PATH,   cols=5)   # your 5-column scripts
#points_all = load_bin_xyz(BIN_PATH, cols=4) # CSF outputs from your notebook


# Apply the ROI so the quadric only fits the region we care about.
def crop(points, x_range, y_range):
    m = np.ones(len(points), dtype=bool)
    if x_range[0] is not None: m &= points[:, 0] >= x_range[0]
    if x_range[1] is not None: m &= points[:, 0] <= x_range[1]
    if y_range[0] is not None: m &= points[:, 1] >= y_range[0]
    if y_range[1] is not None: m &= points[:, 1] <= y_range[1]
    return points[m]


ground = crop(points_all, ROI_X, ROI_Y)

print(f"Loaded   : {len(points_all):,} points from {os.path.basename(BIN_PATH)}")
print(f"After ROI: {len(ground):,} points")
print(f"X range  : [{ground[:, 0].min():7.2f}, {ground[:, 0].max():7.2f}] m")
print(f"Y range  : [{ground[:, 1].min():7.2f}, {ground[:, 1].max():7.2f}] m")
print(f"Z range  : [{ground[:, 2].min():7.2f}, {ground[:, 2].max():7.2f}] m")


## 2. Visualize the point cloud

Three views: top-down (BEV), side (X-Z), and a draggable 3D scatter. All coloured by Z so the hill shape is immediately readable.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 9))

# --- BEV (top-down), coloured by Z ---
sc0 = axes[0].scatter(ground[:, 0], ground[:, 1], c=ground[:, 2],
                      s=2, cmap="terrain")
plt.colorbar(sc0, ax=axes[0], label="Z [m]", shrink=0.8)
axes[0].set_xlabel("X forward [m]")
axes[0].set_ylabel("Y left [m]")
axes[0].set_title("Bird's-eye view — coloured by height")
axes[0].set_aspect("equal")
axes[0].grid(True, lw=0.3)

# --- Side view (X-Z) ---
sc1 = axes[1].scatter(ground[:, 0], ground[:, 2], c=ground[:, 1],
                      s=2, cmap="viridis")
plt.colorbar(sc1, ax=axes[1], label="Y [m]", shrink=0.8)
axes[1].set_xlabel("X forward [m]")
axes[1].set_ylabel("Z height [m]")
axes[1].set_title("Side view — coloured by Y (left/right)")
axes[1].grid(True, lw=0.3)

plt.tight_layout()
plt.show()


# --- 3D scatter, subsampled if large ---
fig = plt.figure(figsize=(10, 7))
ax3d = fig.add_subplot(111, projection="3d")
step = max(1, len(ground) // 20000)
ax3d.scatter(ground[::step, 0], ground[::step, 1], ground[::step, 2],
             s=1, c=ground[::step, 2], cmap="terrain", alpha=0.7)
ax3d.set_xlabel("X [m]"); ax3d.set_ylabel("Y [m]"); ax3d.set_zlabel("Z [m]")
ax3d.set_title("Ground point cloud (3D — drag to rotate)")
plt.tight_layout()
plt.show()


## 3. Fit a global quadric with IRLS

Model: $z = a x^2 + b xy + c y^2 + d x + e y + f$ — six unknowns.

**IRLS in one paragraph.** We start with an ordinary least-squares fit (every point weighted equally). Each point's residual is its distance from the fitted surface. We compute a robust scale estimate from the residuals (the median absolute deviation, MAD), then assign each point a new weight using the **Tukey biweight** function:

$$ w_i = \left(1 - \left(\frac{r_i}{c\,\hat\sigma}\right)^2\right)^2 \quad \text{if } |r_i| < c\,\hat\sigma, \text{ else } w_i = 0 $$

Big residuals (outliers) get tiny weights or zero. We refit using the new weights and iterate until the coefficients stop changing.

**Why this beats OLS for ground curvature.** A single vegetation point that snuck through filtering can pull a least-squares quadric noticeably off. IRLS notices it on iteration 2 and downweights it to ~0; the rest of the points dominate the final fit. Tested on synthetic data with 2% gross outliers, IRLS reduces coefficient error by ~25× compared to OLS.

In [ ]:
def fit_irls_quadric(pts, max_iter=30, tol=1e-7, c_tukey=4.685):
    """Fit z = a x^2 + b xy + c y^2 + d x + e y + f via IRLS (Tukey biweight).

    Returns
    -------
    coeffs    : (6,) array  -> [a, b, c, d, e, f]
    weights   : (N,) array  -> final weights per point in [0, 1]
    residuals : (N,) array  -> z - z_fit at convergence
    n_iter    : int         -> iterations actually run
    """
    x, y, z = pts[:, 0], pts[:, 1], pts[:, 2]
    A = np.column_stack([x**2, x*y, y**2, x, y, np.ones_like(x)])

    # Iteration 0: unweighted LS
    coeffs, *_ = np.linalg.lstsq(A, z, rcond=None)

    weights = np.ones_like(z)
    residuals = z - A @ coeffs

    for it in range(max_iter):
        residuals = z - A @ coeffs
        # Robust scale: 1.4826 * MAD makes it consistent with std for Gaussian data
        scale = 1.4826 * np.median(np.abs(residuals - np.median(residuals)))
        if scale < 1e-12:
            break

        u = residuals / (c_tukey * scale)
        weights = np.where(np.abs(u) < 1, (1 - u**2)**2, 0.0)

        # Weighted least squares: scale rows by sqrt(w)
        sw = np.sqrt(weights)
        Aw = A * sw[:, None]
        zw = z * sw
        new_coeffs, *_ = np.linalg.lstsq(Aw, zw, rcond=None)

        rel_delta = np.linalg.norm(new_coeffs - coeffs) / (np.linalg.norm(coeffs) + 1e-12)
        coeffs = new_coeffs
        if rel_delta < tol:
            return coeffs, weights, residuals, it + 1

    return coeffs, weights, residuals, max_iter


coeffs, weights, residuals, n_iter = fit_irls_quadric(
    ground, max_iter=IRLS_MAX_ITER, tol=IRLS_TOL, c_tukey=IRLS_C_TUKEY,
)
a, b, c, d, e, f = coeffs

print(f"IRLS converged in {n_iter} iterations.")
print(f"  a (x^2)  = {a:+.6f}")
print(f"  b (xy)   = {b:+.6f}")
print(f"  c (y^2)  = {c:+.6f}")
print(f"  d (x)    = {d:+.6f}")
print(f"  e (y)    = {e:+.6f}")
print(f"  f (1)    = {f:+.6f}")
print()
print(f"Inlier fraction (w > 0.05): {(weights > 0.05).mean():.3f}")
print(f"Residual RMSE (weighted)   : "
      f"{np.sqrt(np.average(residuals**2, weights=weights)):.4f} m")
print(f"Residual MAD               : "
      f"{1.4826 * np.median(np.abs(residuals - np.median(residuals))):.4f} m")


### Visualize the fitted quadric

Three plots:
1. 3D — point cloud + the fitted quadric surface (transparent mesh).
2. BEV scatter coloured by **signed residual** (z - z_fit). Patches above the surface go red, below go blue. Helps you spot anything systematic the quadric is missing.
3. Histogram of residuals + the IRLS weights so you can see what got downweighted.

In [ ]:
def quadric_z(coeffs, x, y):
    a, b, c, d, e, f = coeffs
    return a*x**2 + b*x*y + c*y**2 + d*x + e*y + f


# --- 3D: points + fitted surface ---
xg = np.linspace(ground[:, 0].min(), ground[:, 0].max(), 40)
yg = np.linspace(ground[:, 1].min(), ground[:, 1].max(), 40)
Xg, Yg = np.meshgrid(xg, yg)
Zg = quadric_z(coeffs, Xg, Yg)

fig = plt.figure(figsize=(11, 7))
ax3d = fig.add_subplot(111, projection="3d")
step = max(1, len(ground) // 12000)
ax3d.scatter(ground[::step, 0], ground[::step, 1], ground[::step, 2],
             s=1, c="tab:gray", alpha=0.5, label="ground points")
ax3d.plot_surface(Xg, Yg, Zg, alpha=0.45, cmap="terrain",
                  edgecolor="none", linewidth=0)
ax3d.set_xlabel("X [m]"); ax3d.set_ylabel("Y [m]"); ax3d.set_zlabel("Z [m]")
ax3d.set_title("Ground points + IRLS quadric (drag to rotate)")
plt.tight_layout()
plt.show()


# --- BEV residual map ---
vmax = float(np.percentile(np.abs(residuals), 95))
fig, ax = plt.subplots(figsize=(11, 6))
sc = ax.scatter(ground[:, 0], ground[:, 1], c=residuals,
                s=3, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
plt.colorbar(sc, ax=ax, label="z - z_fit [m]")
ax.set_xlabel("X [m]"); ax.set_ylabel("Y [m]")
ax.set_title("Residuals (signed) — point minus quadric prediction")
ax.set_aspect("equal"); ax.grid(True, lw=0.3)
plt.tight_layout()
plt.show()


# --- residual histogram + weight distribution ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(residuals, bins=80, color="tab:gray", edgecolor="white")
axes[0].axvline(0, color="k", lw=0.7)
axes[0].set_xlabel("Residual z - z_fit [m]")
axes[0].set_ylabel("Count")
axes[0].set_title("Residual distribution")
axes[0].grid(True, lw=0.3)

axes[1].hist(weights, bins=40, color="tab:purple", edgecolor="white")
axes[1].set_xlabel("IRLS weight  (1 = inlier, 0 = outlier)")
axes[1].set_ylabel("Count")
axes[1].set_title("Final IRLS weights")
axes[1].grid(True, lw=0.3)
plt.tight_layout()
plt.show()


## 4. Curvature from the quadric — analytic, no numerical differentiation

For a height-field surface $z = f(x, y)$, mean and Gaussian curvature have closed forms in terms of the partial derivatives. With our quadric:

$$ f_x = 2ax + by + d, \quad f_y = bx + 2cy + e $$
$$ f_{xx} = 2a, \quad f_{xy} = b, \quad f_{yy} = 2c $$

(Note: second derivatives are constants, but the curvature still varies across the surface because the denominator depends on the slope.)

Mean curvature:

$$ H(x,y) = \frac{(1+f_y^2)f_{xx} - 2 f_x f_y f_{xy} + (1+f_x^2) f_{yy}}{2\,(1 + f_x^2 + f_y^2)^{3/2}} $$

Gaussian curvature:

$$ K(x,y) = \frac{f_{xx} f_{yy} - f_{xy}^2}{(1 + f_x^2 + f_y^2)^2} $$

Sign conventions: with the surface normal pointing **up** (which our quadric's $z$ implicitly assumes), $H < 0$ at hill crests (concave down from above), $H > 0$ in valley bottoms. $K > 0$ for bowl/dome shapes (both principal curvatures same sign), $K < 0$ for saddles, $K = 0$ if the surface is flat in at least one direction (a ramp or cylinder).

In [ ]:
def curvature_at(coeffs, x, y):
    """Mean and Gaussian curvature at (x, y) for the global quadric."""
    a, b, c, d, e, f = coeffs
    fx  = 2*a*x + b*y + d
    fy  = b*x + 2*c*y + e
    fxx, fxy, fyy = 2*a, b, 2*c
    denom = 1.0 + fx**2 + fy**2
    H = ((1 + fy**2)*fxx - 2*fx*fy*fxy + (1 + fx**2)*fyy) / (2 * denom**1.5)
    K = (fxx*fyy - fxy**2) / denom**2
    return H, K


# Curvature at every original point
H_all, K_all = curvature_at(coeffs, ground[:, 0], ground[:, 1])

# Summary
print(f"Mean curvature (H):  min={H_all.min():+.4f}  median={np.median(H_all):+.4f}  max={H_all.max():+.4f}  [1/m]")
print(f"Gaussian curvature (K): min={K_all.min():+.5f}  median={np.median(K_all):+.5f}  max={K_all.max():+.5f}  [1/m^2]")
print()
print("Interpretation cheat sheet:")
print("  H < 0  -> concave down (hill crest)")
print("  H > 0  -> concave up   (valley bottom)")
print("  K > 0  -> bowl or dome")
print("  K < 0  -> saddle")
print("  K ~ 0  -> ramp or cylinder")


# BEV map of mean curvature
vlim = float(np.percentile(np.abs(H_all), 98))
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sc1 = axes[0].scatter(ground[:, 0], ground[:, 1], c=H_all,
                      s=3, cmap="RdBu_r", vmin=-vlim, vmax=vlim)
plt.colorbar(sc1, ax=axes[0], label=r"Mean curvature  H  [1/m]")
axes[0].set_xlabel("X [m]"); axes[0].set_ylabel("Y [m]")
axes[0].set_title("Mean curvature (signed)")
axes[0].set_aspect("equal"); axes[0].grid(True, lw=0.3)

vlim_K = float(np.percentile(np.abs(K_all), 98))
sc2 = axes[1].scatter(ground[:, 0], ground[:, 1], c=K_all,
                      s=3, cmap="PuOr_r", vmin=-vlim_K, vmax=vlim_K)
plt.colorbar(sc2, ax=axes[1], label=r"Gaussian curvature  K  [1/m^2]")
axes[1].set_xlabel("X [m]"); axes[1].set_ylabel("Y [m]")
axes[1].set_title("Gaussian curvature (signed)")
axes[1].set_aspect("equal"); axes[1].grid(True, lw=0.3)

plt.tight_layout()
plt.show()


## 5. Forward 1-D height profile

Same layout as your friend's `plot_road_profile`, top panel: the bin-median heights as scattered points + the smooth quadric prediction along the strip. Slice direction = X (forward), strip = $|y - y_0| \le$ `PROFILE_Y_HALF`.

The quadric line is the truth-of-the-fit; the dots are what the raw data look like at half-metre granularity.

In [ ]:
# 1. crop a forward strip around the centreline
strip_mask = (
    (ground[:, 0] >= PROFILE_X_RANGE[0]) &
    (ground[:, 0] <= PROFILE_X_RANGE[1]) &
    (np.abs(ground[:, 1] - PROFILE_Y_CENTER) <= PROFILE_Y_HALF)
)
strip_pts = ground[strip_mask]
print(f"Profile strip: {len(strip_pts):,} points")

# 2. bin medians (only for visualization -- the quadric is the actual model)
x_edges   = np.arange(PROFILE_X_RANGE[0], PROFILE_X_RANGE[1] + PROFILE_BIN, PROFILE_BIN)
x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])
z_bin = np.full_like(x_centers, np.nan)
for i in range(len(x_centers)):
    in_bin = (strip_pts[:, 0] >= x_edges[i]) & (strip_pts[:, 0] < x_edges[i + 1])
    if in_bin.sum() >= 5:
        z_bin[i] = float(np.median(strip_pts[in_bin, 2]))

# 3. quadric prediction along the strip centreline (y = PROFILE_Y_CENTER)
x_smooth = np.linspace(PROFILE_X_RANGE[0], PROFILE_X_RANGE[1], 400)
z_smooth = quadric_z(coeffs, x_smooth, PROFILE_Y_CENTER)

# Keep these for the curvature panel below
profile = dict(
    x_centers=x_centers, z_bin=z_bin,
    x_smooth=x_smooth, z_smooth=z_smooth,
)


## 6. Forward 1-D signed curvature graph

Bottom panel of your friend's plot. Along the strip (slicing the quadric at $y = y_0$), the height as a function of $x$ alone is

$$ z(x) = a x^2 + (b y_0 + d) x + (c y_0^2 + e y_0 + f) $$

so the slope is $z'(x) = 2 a x + b y_0 + d$ and the second derivative is constantly $2a$. The signed Frenet curvature (same formula your friend uses) is then

$$ \kappa(x) = \frac{z''(x)}{(1 + z'(x)^2)^{3/2}} = \frac{2a}{\bigl(1 + (2 a x + b y_0 + d)^2\bigr)^{3/2}} $$

so $\kappa$ varies smoothly with $x$ (the slope changes, the denominator changes), even though the second derivative itself doesn't. Sign convention is the same: $\kappa > 0$ = valley bottom, $\kappa < 0$ = hill crest.

The next cell renders the full three-panel plot — height, slope, signed curvature — in your friend's layout, with the quadric replacing his per-bin PCA.

In [ ]:
# Slice the quadric at y = PROFILE_Y_CENTER for slope and curvature
y0 = PROFILE_Y_CENTER

dzdx_smooth   = 2*a*x_smooth + b*y0 + d
slope_deg     = np.degrees(np.arctan(dzdx_smooth))
d2zdx2_smooth = np.full_like(x_smooth, 2*a)
kappa_smooth  = d2zdx2_smooth / (1.0 + dzdx_smooth**2)**1.5

# Raw bin slopes for the green dots in the slope panel
# (quick finite difference between adjacent z-bin medians, just for context)
valid = ~np.isnan(z_bin)
dzdx_raw = np.full_like(x_centers, np.nan)
if valid.sum() >= 2:
    g = np.gradient(np.where(valid, z_bin, np.nan), PROFILE_BIN)
    dzdx_raw = g

slope_raw_deg = np.degrees(np.arctan(dzdx_raw))


fig, ax = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

# --- 1. height ---
ax[0].plot(x_smooth, z_smooth, "-", lw=1.5, color="tab:blue",
           label="z(x) from IRLS quadric")
ax[0].plot(x_centers[valid], z_bin[valid], "o", ms=4, color="tab:orange",
           label="bin medians (raw)")
ax[0].set_ylabel("Ground height z  [m]")
ax[0].set_title("Forward road profile -- IRLS quadric on ground points")
ax[0].grid(True); ax[0].legend()

# --- 2. slope (signed) ---
ax[1].plot(x_smooth, slope_deg, "-", color="tab:green",
           lw=1.5, label="slope from quadric")
ax[1].plot(x_centers[~np.isnan(slope_raw_deg)],
           slope_raw_deg[~np.isnan(slope_raw_deg)],
           "o", ms=3, color="tab:olive", alpha=0.5,
           label="raw bin-to-bin")
ax[1].axhline(0, color="k", lw=0.5)
ax[1].set_ylabel("Slope [deg]  (signed)")
ax[1].grid(True); ax[1].legend()

# --- 3. signed curvature, fill-coloured by sign ---
k = kappa_smooth
pos = k >= 0
ax[2].fill_between(x_smooth, k, 0, where=pos, color="tab:blue", alpha=0.35,
                   label=r"$\kappa > 0$  (valley bottom)")
ax[2].fill_between(x_smooth, k, 0, where=~pos, color="tab:red", alpha=0.35,
                   label=r"$\kappa < 0$  (hill crest)")
ax[2].plot(x_smooth, k, "-", color="k", lw=1)
ax[2].axhline(0, color="k", lw=0.5)
ax[2].set_xlabel("X (forward) [m]")
ax[2].set_ylabel(r"Curvature $\kappa$ [1/m]  (signed)")
ax[2].grid(True); ax[2].legend(loc="upper right")

plt.tight_layout(); plt.show()


# Numerical summary -- same fields as your friend's print-out
print(f"Method        : IRLS quadric (global)")
print(f"Slice y       : {y0:+.2f} m")
print(f"Mean slope    : {slope_deg.mean():+6.2f}  deg")
print(f"Mean kappa    : {k.mean():+7.4f}  1/m")
print(f"Max |kappa|   : {np.abs(k).max():7.4f}  1/m")
if np.abs(k).max() > 1e-6:
    print(f"Min radius    : {1.0 / np.abs(k).max():7.1f}  m")
n_pos = int((k > 0).sum())
n_neg = int((k < 0).sum())
print(f"Concave-up    : {n_pos:4d} samples  (valley-shaped)")
print(f"Concave-down  : {n_neg:4d} samples  (hill-shaped)")


## When to abandon the global quadric

A single quadric works well when the ground in your ROI is approximately one of: a planar slope, a single dome, a single bowl, or a saddle. It will fail (give very low residuals everywhere but visibly miss real features) when:

- the ground has multiple bumps or undulations within the ROI;
- there's a sharp break in slope (cliff edge, retaining wall remnant, road crown);
- the surface bends in more than one way along the same axis (e.g. a hill followed by a valley along X).

If you see large systematic patterns in the **residual BEV** in §3 — coherent regions of red and blue rather than salt-and-pepper noise — that's the diagnostic. The fix is one of:

1. **Tighten the ROI** so you're fitting only one feature at a time.
2. **Higher-order surface** — switch to a bicubic ($z$ as a polynomial of total degree 3) or to *jet fitting* (Cazals–Pouget). The IRLS scaffold above is unchanged; only the design matrix `A` grows columns.
3. **Local IRLS quadrics** — for each ground point, fit an IRLS quadric to its k-nearest neighbours and read curvature locally. This is what method 16 (Kalogerakis et al. 2009) literally is. Heavier to compute but it gives you a curvature *field* rather than a single smooth function. This is the cleanest like-for-like comparison to your friend's per-bin PCA.

Drop me a note when you're ready to compare against your friend's PCA result on the same scan — I can build a side-by-side overlay plot.